Import current pipeline timing CSV

In [21]:
import pandas as pd

long_term_memory = False

# Chargement du fichier CSV
df = pd.read_csv('../../experiments/current/pipeline_timing.csv')

colonnes_not_nan = df.columns[df.isna().all()==False]
df.drop(columns=['agent_id'], inplace=True)

if long_term_memory == False:
    df.drop(columns=['T_ltm_start', 'T_ltm_end'], inplace=True)


print("Na Columns: ")
print(df.columns[df.isna().all()])


print(df.isna().sum())


Na Columns: 
Index([], dtype='str')
sim_time                  0
T0                        0
T_parse                   0
T_flag                    0
T_otp_start               0
T_transit_sem           264
T_transit_end           264
T_osmnx_sem            1537
T_osmnx_end              15
T_otp_end                15
T_llm_start              26
T_llm_sent               26
T_llm_result             26
T_extract_end            54
T_enqueue                15
T_fin                     0
P4_4_ms                1779
P5_1_ms                1779
P5_3_ms                1779
P5_4_ms                1779
P5_5_ms                1779
P5_llm_provider        1779
P5_llm_retries            0
P5_tokens_in              0
P5_tokens_out             0
plan_selected_index      15
selection_method         15
dtype: int64


Extract Date columns in df_delay

In [22]:
df_delay = df.drop(columns=['P4_4_ms', 'P5_1_ms', 'P5_3_ms', 'P5_4_ms',
       'P5_5_ms', 'P5_llm_provider', 'P5_llm_retries', 'P5_tokens_in',
       'P5_tokens_out'])

df_delay

,sim_time,T0,T_parse,T_flag,T_otp_start,T_transit_sem,T_transit_end,T_osmnx_sem,T_osmnx_end,T_otp_end,T_llm_start,T_llm_sent,T_llm_result,T_extract_end,T_enqueue,T_fin,plan_selected_index,selection_method
0,1775797200,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,NaN,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,0.0,LLM
1,1775801700,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,NaN,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,0.0,LLM
2,1775803500,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,NaN,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,0.0,LLM
3,1775804400,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,NaN,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,0.0,LLM
4,1775804400,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,1.778853e+09,0.0,LLM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1957,1775921400,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,NaN,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,0.0,LLM
1958,1775922300,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,NaN,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,2.0,LLM
1959,1775922300,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,NaN,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,2.0,LLM
1960,1775922300,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,1.778855e+09,2.0,LLM


Compute delays

In [23]:
df_delay["parse"]        = df_delay["T_parse"]       - df_delay["T0"]
df_delay["flag"]         = df_delay["T_flag"]        - df_delay["T_parse"]
df_delay["gap_otp"]      = df_delay["T_otp_start"]   - df_delay["T_flag"]
df_delay["transit_sem"]  = df_delay["T_transit_sem"] - df_delay["T_otp_start"]  # 3A queue
df_delay["transit_req"]  = df_delay["T_transit_end"] - df_delay["T_transit_sem"] # 3A HTTP
df_delay["transit"]      = df_delay["T_transit_end"] - df_delay["T_otp_start"]   # 3A total
df_delay["osmnx_sem"]    = df_delay["T_osmnx_sem"]   - df_delay["T_otp_start"]  # 3B queue (HTTP mode)
df_delay["osmnx_req"]    = df_delay["T_osmnx_end"]   - df_delay["T_osmnx_sem"]  # 3B processing (HTTP mode)
df_delay["osmnx"]        = df_delay["T_osmnx_end"]   - df_delay["T_otp_start"]  # 3B total
df_delay["otp"]          = df_delay["T_otp_end"]     - df_delay["T_otp_start"]  # 3 total = max(3A, 3B)
df_delay["gap_llm"]      = df_delay["T_llm_start"]   - df_delay["T_otp_end"]    # 3→5 transition
df_delay["llm_post"]     = df_delay["T_llm_sent"]    - df_delay["T_llm_start"]
df_delay["llm_wait"]     = df_delay["T_llm_result"]  - df_delay["T_llm_sent"]
df_delay["extract"]      = df_delay["T_extract_end"] - df_delay["T_llm_result"]
df_delay["enqueue"]      = df_delay["T_enqueue"]     - df_delay["T_extract_end"]
df_delay["ws"]           = df_delay["T_fin"]         - df_delay["T_enqueue"]
df_delay["total"]        = df_delay["T_fin"]         - df_delay["T0"]

if long_term_memory:
    df_delay["ltm"] = df_delay["T_ltm_end"] - df_delay["T_ltm_start"]

timestamp_cols = ["T_parse", "T_flag", "T_otp_start",
                  "T_transit_sem", "T_transit_end", "T_osmnx_sem", "T_osmnx_end", "T_otp_end",
                  "T_llm_start", "T_llm_sent", "T_llm_result", "T_extract_end", "T_enqueue"]
if long_term_memory:
    timestamp_cols += ["T_ltm_start", "T_ltm_end"]
df_delay = df_delay.drop(columns=[c for c in timestamp_cols if c in df_delay.columns])
df_delay

,sim_time,T0,T_fin,plan_selected_index,selection_method,parse,flag,gap_otp,transit_sem,transit_req,...,osmnx_req,osmnx,otp,gap_llm,llm_post,llm_wait,extract,enqueue,ws,total
0,1775797200,1.778853e+09,1.778853e+09,0.0,LLM,0.000046,0.003867,0.001009,0.002390,0.188928,...,NaN,0.006160,0.202922,0.002797,0.04317,16.660782,0.029607,0.039852,0.535144,17.519195
1,1775801700,1.778853e+09,1.778853e+09,0.0,LLM,0.000038,0.003052,0.000935,0.000921,0.505442,...,NaN,0.001467,0.530649,0.000883,0.08181,1.081951,0.041170,0.065661,0.253787,2.059935
2,1775803500,1.778853e+09,1.778853e+09,0.0,LLM,0.000046,0.007890,0.005619,0.000525,0.227351,...,NaN,0.001093,0.255428,0.002306,0.04566,0.589380,0.036124,0.091324,0.556200,1.589975
3,1775804400,1.778853e+09,1.778853e+09,0.0,LLM,0.000248,0.004391,0.001783,0.006896,0.204572,...,NaN,0.007727,0.232973,0.000524,0.07692,1.511636,0.088146,0.228309,0.001156,2.146087
4,1775804400,1.778853e+09,1.778853e+09,0.0,LLM,0.000248,0.004391,0.001256,0.000944,1.202677,...,3.899368,3.906683,3.921202,0.005138,0.04574,1.128101,0.019727,0.061710,0.000648,5.188161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1957,1775921400,1.778855e+09,1.778855e+09,0.0,LLM,0.000473,0.040896,0.177575,0.023970,3.012218,...,NaN,0.024868,3.073640,0.000591,0.43828,13.218125,0.095079,0.078964,0.495765,17.619389
1958,1775922300,1.778855e+09,1.778855e+09,2.0,LLM,0.000063,0.023494,0.007232,0.009794,1.212432,...,NaN,0.010306,1.256308,0.000432,0.20660,7.121905,0.006139,0.212608,0.945132,9.779914
1959,1775922300,1.778855e+09,1.778855e+09,2.0,LLM,0.000063,0.023494,0.009397,0.099061,1.120697,...,NaN,0.102112,1.244048,0.000547,0.22212,7.482862,0.034763,0.194130,0.570308,9.781734
1960,1775922300,1.778855e+09,1.778855e+09,2.0,LLM,0.000063,0.023494,0.009136,0.014471,1.205284,...,7.334221,7.433446,7.515543,0.000785,0.06210,1.600429,0.156075,0.250442,0.164661,9.782729


In [24]:
delay_cols_ordered = [
    "parse",        # 1
    "flag",         # 2
    "gap_otp",      # 2→3
    "otp",          # 3 total
    "transit",      # 3A total
    "transit_sem",  # 3A-queue
    "transit_req",  # 3A-calc
    "osmnx",        # 3B total
    "osmnx_sem",    # 3B-queue (HTTP mode)
    "osmnx_req",    # 3B-calc (HTTP mode)
    "gap_llm",      # 3→5
    "llm_post",     # 5
    "llm_wait",     # 6
    "extract",      # 7
    "enqueue",      # 8
    "ws",           # 9
    "total",
]
if long_term_memory:
    delay_cols_ordered.insert(delay_cols_ordered.index("llm_post"), "ltm")

delay_cols_ordered = [c for c in delay_cols_ordered if c in df_delay.columns]

descriptions = {
    "parse":       "1     — Parsing JSON + validation Pydantic de la requête GAMA",
    "flag":        "2     — Filtrage des agents éligibles (scan de toute la population)",
    "gap_otp":     "2→3   — Délai asyncio entre lancement des tâches et démarrage OTP",
    "otp":         "3     — OTP + OSMnx gather total : durée = max(3A, 3B)",
    "transit":     "3A    — OTP transit total (queue + requête HTTP GraphQL)",
    "transit_sem": "3A-q  — Attente sémaphore OTP (queue, avant accès HTTP)",
    "transit_req": "3A-c  — Requête HTTP GraphQL OTP (calcul itinéraire transit)",
    "osmnx":       "3B    — OSMnx pied/vélo/voiture total (parallèle avec 3A)",
    "osmnx_sem":   "3B-q  — Attente sémaphore OSMnx (HTTP mode uniquement)",
    "osmnx_req":   "3B-c  — Calcul OSMnx après sémaphore (HTTP ou process pool)",
    "gap_llm":     "3→5   — Sélection candidats + construction payload LLM (non instrumenté)",
    "ltm":         "4     — Requête ChromaDB mémoire long terme (si LTM activée)",
    "llm_post":    "5     — HTTP POST création de la tâche dans le gateway LLM",
    "llm_wait":    "6     — Attente résultat LLM : long-poll Pub/Sub (micro-batch + worker + inférence)",
    "extract":     "7     — Extraction chosen_index + remapping vers le plan original (post-shuffle)",
    "enqueue":     "8     — Construction PersonMove + MoveLogger I/O + enqueue",
    "ws":          "9     — Attente envoi WebSocket vers GAMA (publish_loop)",
    "total":       "TOTAL — Durée totale pipeline T0 → T_fin",
}

delay_mean_df = df_delay[delay_cols_ordered].describe().loc[["mean", "std", "min", "max"]].T
delay_mean_df["Description"] = delay_mean_df.index.map(descriptions)
delay_mean_df.style.format("{:.4f}", subset=["mean", "std", "min", "max"])

,mean,std,min,max,Description
parse,0.0003,0.0004,0.0000,0.0021,1 — Parsing JSON + validation Pydantic de la requête GAMA
flag,0.0119,0.0096,0.0030,0.0474,2 — Filtrage des agents éligibles (scan de toute la population)
gap_otp,0.0580,0.1281,0.0006,0.9494,2→3 — Délai asyncio entre lancement des tâches et démarrage OTP
otp,4.7421,5.8245,0.0010,48.1282,"3 — OTP + OSMnx gather total : durée = max(3A, 3B)"
transit,2.4376,1.6337,0.0189,10.4108,3A — OTP transit total (queue + requête HTTP GraphQL)
transit_sem,0.6446,1.2003,0.0005,6.6439,"3A-q — Attente sémaphore OTP (queue, avant accès HTTP)"
transit_req,1.7930,1.2294,0.0184,7.9675,3A-c — Requête HTTP GraphQL OTP (calcul itinéraire transit)
osmnx,2.8491,6.3871,0.0008,48.1228,3B — OSMnx pied/vélo/voiture total (parallèle avec 3A)
osmnx_sem,4.8348,6.3863,0.0011,33.4163,3B-q — Attente sémaphore OSMnx (HTTP mode uniquement)
osmnx_req,8.0767,3.3083,1.6782,19.8263,3B-c — Calcul OSMnx après sémaphore (HTTP ou process pool)


Véfification que l'on a toutes les valeurs

In [25]:
# otp = total gather (transit+osmnx parallèles), transit/osmnx sont des sous-composantes
# gap_llm ferme le trou entre OTP et LLM
segment_cols = ["parse", "flag", "gap_otp", "otp", "gap_llm", "llm_post", "llm_wait", "extract", "enqueue", "ws"]
if long_term_memory:
    segment_cols.insert(segment_cols.index("llm_post"), "ltm")
segment_cols = [c for c in segment_cols if c in df_delay.columns]

df_llm = df_delay[df_delay["selection_method"] == "LLM"].copy()
df_llm["sum_segments"] = df_llm[segment_cols].sum(axis=1)
df_llm["diff"] = df_llm["total"] - df_llm["sum_segments"]

print(f"Lignes vérifiées (selection_method=LLM) : {len(df_llm)}")
print(f"Diff max  : {df_llm['diff'].abs().max():.4f} s")
print(f"Diff mean : {df_llm['diff'].abs().mean():.4f} s")
print()
df_llm[["total", "sum_segments", "diff"]].describe().style.format("{:.4f}")

Lignes vérifiées (selection_method=LLM) : 1908
Diff max  : 0.0000 s
Diff mean : 0.0000 s



,total,sum_segments,diff
count,1908.0000,1908.0000,1908.0000
mean,15.1373,15.1373,0.0000
std,10.4041,10.4041,0.0000
min,1.2717,1.2717,0.0000
25%,8.2553,8.2553,0.0000
50%,12.4568,12.4568,0.0000
75%,18.1281,18.1281,0.0000
max,93.2213,93.2213,0.0000
